# LTR + Prompt Length Extension — Full Pipeline (v3, rebuilt)

This notebook was rebuilt from scratch after several real failures in
earlier sessions: lost checkpoints (working from `/content` instead of
Drive), package version conflicts (vLLM pulling incompatible
`transformers`/`tokenizers`), and a model-loading mismatch (custom
`LTRPredictor` architecture being loaded with the wrong HF class).

**All three are fixed structurally in this version:**
- Drive is mounted in Cell 1, before anything else. Every later cell
  works inside the Drive-mounted path, not `/content`.
- Package versions are pinned ONCE, in the correct install order, so
  nothing gets silently upgraded by a later `pip install`.
- The loader code (already fixed in `2_kendalls_tau_eval.py` and
  `3_vllm_benchmark.py`) matches the actual `LTRPredictor` class.

**Before running:** Runtime → Change runtime type → select a GPU
(T4 is fine for training; switch to A100 only for Section 10 onward,
the vLLM benchmark, if you have compute units to spend there).

**v3 update:** Section 3 now clones
[`github.com/waizinmon/prompt-aware-ltr`](https://github.com/waizinmon/prompt-aware-ltr)
directly instead of `hao-ai-lab/vllm-ltr` + manual upload. Scripts live in
`scripts/` now, so every `!python ...` call below points there, and Section 7 is
now a presence-check instead of a `files.upload()` step -- no more re-uploading
files each session.

## 1. Mount Google Drive (FIRST — before anything else)

Everything in this notebook lives under `/content/drive/MyDrive/ltr_extension`
from this point forward. This directory survives session restarts,
disconnects, and runtime changes — `/content` on its own does not.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/ltr_extension'
os.makedirs(PROJECT_DIR, exist_ok=True)
%cd {PROJECT_DIR}
!pwd

## 2. Confirm GPU is attached

In [ ]:
!nvidia-smi
import torch
print()
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Clone your prompt-aware-ltr repo (into the Drive-mounted path)

Clones [github.com/waizinmon/prompt-aware-ltr](https://github.com/waizinmon/prompt-aware-ltr) -- your
extension code (`scripts/`, `data/`, `checkpoints/`), not the vLLM engine itself.
vLLM is installed separately via `pip install vllm` in the next step. If this is
a re-run and the folder already exists from a previous session, this cell skips
the clone safely instead of erroring.

**If the repo is private:** a plain `git clone` will fail since it can't answer
an auth prompt. Either make the repo public, or clone with a token in the URL:
`https://<github_username>:<personal_access_token>@github.com/waizinmon/prompt-aware-ltr.git`.

In [ ]:
import os
repo_path = os.path.join(PROJECT_DIR, 'prompt-aware-ltr')

if os.path.exists(repo_path):
    print(f"Repo already exists at {repo_path}, skipping clone.")
else:
    !git clone https://github.com/waizinmon/prompt-aware-ltr.git {repo_path}

%cd {repo_path}
!pwd
!ls

## 4. Install everything in ONE pinned step (fixes prior version conflicts)

**Why this is different from before:** earlier sessions installed vLLM
first, which silently pulled in a `transformers`/`tokenizers` combo
that later broke. This time, all versions are pinned together in one
command, in an order that's been confirmed to work together:

- `vllm==0.5.4` -- the original paper's vllm==0.4.1 has no Python 3.12
  wheel (Colab's current default), and building it from source fails.
  0.5.4 is the closest version with stable Python 3.12 support.
- `transformers==4.44.2` + `tokenizers==0.19.1` -- pinned TOGETHER so
  they're compatible with each other (the earlier `--no-deps` fix was
  incomplete and caused a second conflict).

This is a single cell, run once. Do not re-run `pip install vllm`
separately later -- that would re-trigger the same conflict.

**Update (corrected):** vLLM 0.5.4's guided-decoding feature depends on a PyPI package called `pyairports` for airport-code validation inside `outlines`. That package turns out to be broken -- pip reports it as installed, but `import pyairports` fails with `ModuleNotFoundError` regardless, because the package contains no actual code. Pinning specific `outlines`/`pyairports` versions does not fix this. The reliable fix is installing current stable vLLM instead of pinning to 0.5.4 -- newer vLLM releases no longer hit this broken dependency path.

**Full third-party dependency set for this pipeline:** `vllm` (pulls in a
compatible `torch` automatically -- don't pin `torch` separately, that's what
caused conflicts before), `transformers==4.44.2`, `tokenizers==0.19.1`,
`datasets`, `scipy`, `matplotlib`, `huggingface_hub` (for the login/token flow
in Sections 6 and 8), and `scikit-learn` (kept for compatibility, not required by
the current scripts). Everything else the scripts import (`argparse`, `json`,
`os`, `random`, `itertools`, `typing`, `dataclasses`, `asyncio`, `time`,
`collections`, `sys`, `importlib`) is Python standard library -- nothing more to
install.

In [ ]:
# vLLM 0.5.4's guided-decoding feature pulls in `outlines`, which depends
# on a PyPI package literally named "pyairports" for airport-code
# validation. That PyPI package turned out to be broken/placeholder --
# pip installs it successfully, but `import pyairports` fails immediately
# because it contains no real module inside. Rather than fight that,
# we sidestep it entirely by installing current stable vLLM, which no
# longer has this fragile dependency chain.
!pip install -q "transformers==4.44.2" "tokenizers==0.19.1" vllm datasets scipy matplotlib scikit-learn huggingface_hub

print()
print("Installed versions:")
import transformers, tokenizers
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
import vllm
print("vllm:", vllm.__version__)
print()
print("NOTE: original paper used vllm==0.4.1. This environment uses a")
print("newer version due to Python 3.12 compatibility AND a broken")
print("transitive dependency (pyairports) in the 0.5.x guided-decoding")
print("feature. State this version deviation explicitly in your report.")

## 5. RESTART RUNTIME NOW (required once, after Cell 4)

`transformers` may already be partially imported in memory from the
install process. A restart clears that cleanly.

**Runtime → Restart session**

After restarting, **skip Cells 1-4** (Drive is still mounted, packages
are still installed -- a restart clears Python memory, not installed
packages or Drive files) and continue from Cell 6 onward.

In [ ]:
# Run this AFTER restarting, to confirm you're back in the right place
# and everything from Cells 1-4 survived the restart correctly.
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/ltr_extension'
repo_path = os.path.join(PROJECT_DIR, 'prompt-aware-ltr')
%cd {repo_path}
!pwd

import transformers, tokenizers, vllm
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("vllm:", vllm.__version__)
print("\nIf the versions above match Cell 4's output, you're good to continue.")

## 6. HuggingFace login

Needed for: (a) downloading the gated LMSYS-Chat-1M dataset, and
(b) downloading the gated Llama-3.1-8B-Instruct model later.

Before running this cell:
1. Accept the dataset terms: https://huggingface.co/datasets/lmsys/lmsys-chat-1m
2. Accept the model terms: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
3. Get a token (read access is enough): https://huggingface.co/settings/tokens

**Paste your token directly into the login() call below just this once.**
After this cell runs, every later cell reuses this login automatically
via `HfFolder.get_token()` -- you will not need to paste it again.

In [ ]:
from huggingface_hub import login
login(token="PASTE_YOUR_TOKEN_HERE")  # <-- replace this string only, then run

## 7. Verify the project scripts are present

Since Section 3 now clones `prompt-aware-ltr` directly, the pipeline scripts
already live in `scripts/` -- no manual upload needed.

**Required for the core pipeline (Sections 8-14):**
```
scripts/0a_build_dataset.py
scripts/0b_train_predictor.py
scripts/1_scoring_patch.py
scripts/scoring_patch_helper.py
scripts/3_vllm_benchmark.py
scripts/4_plot_results.py
```

**Optional** -- only needed for the bonus Kendall's Tau cell near the end of
this notebook (Section "Optional — Kendall's Tau evaluation"):
```
scripts/2_kendalls_tau_eval.py
```

In [ ]:
import os
required = ["0a_build_dataset.py", "0b_train_predictor.py", "1_scoring_patch.py",
            "scoring_patch_helper.py", "3_vllm_benchmark.py", "4_plot_results.py"]
optional = ["2_kendalls_tau_eval.py"]

scripts_dir = os.path.join(repo_path, "scripts")
for fname in required + optional:
    path = os.path.join(scripts_dir, fname)
    tag = "REQUIRED" if fname in required else "optional"
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"[{status}] ({tag}) {fname}")


## 8. Build the dataset

Downloads the real public LMSYS-Chat-1M dataset and splits it into
training data + two evaluation sets. Dataset 2 deliberately samples
from the extremes of the output-length distribution to create genuine
distribution shift, matching what the original paper's Dataset 2 was
designed to expose.

**Note:** this reconstructs an equivalent dataset from the same public
source the original paper cites -- it is not their exact unpublished
23,800-sample file. Report results as "evaluated on a reconstructed
LMSYS-Chat-1M split," not as an exact reproduction.

This step is skipped automatically if the files already exist from a
previous run in this session.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
PROJECT_DIR = '/content/drive/MyDrive/ltr_extension'
repo_path = os.path.join(PROJECT_DIR, 'prompt-aware-ltr')
%cd {repo_path}

import os
from huggingface_hub import HfFolder

data_dir = os.path.join(repo_path, 'data')
expected_files = ['train_predictor.jsonl', 'dataset1_eval.jsonl', 'dataset2_eval.jsonl']
already_built = os.path.exists(data_dir) and all(
    os.path.exists(os.path.join(data_dir, f)) for f in expected_files
)

if already_built:
    print(f"Dataset files already exist in {data_dir}, skipping build.")
    !ls -la {data_dir}
else:
    token = HfFolder.get_token()
    !python scripts/0a_build_dataset.py \
        --hf_token {token} \
        --train_size 23800 \
        --eval_size 1000 \
        --output_dir ./data

## 9. Train the predictor — TWO checkpoints

Trains both the original paper's baseline (ListMLE loss) and your
Extension #6 (pairwise margin ranking loss). Both checkpoints are
needed for a fair four-way comparison later.

**Each cell below checks if its checkpoint already exists and skips
training if so** -- safe to re-run this whole notebook without
wasting compute on checkpoints you already have.

The model architecture (`LTRPredictor`, defined in
`0b_train_predictor.py`) is a custom OPT-125M + linear score head --
NOT a standard HF classification model. The benchmark and evaluation
scripts already account for this; don't swap in
`AutoModelForSequenceClassification` anywhere, that was the bug that
caused the `config.json` errors in earlier sessions.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
PROJECT_DIR = '/content/drive/MyDrive/ltr_extension'
repo_path = os.path.join(PROJECT_DIR, 'prompt-aware-ltr')
%cd {repo_path}

import os

ckpt1 = os.path.join(repo_path, 'checkpoints', 'opt125m-ltr-original')
if os.path.exists(os.path.join(ckpt1, 'pytorch_model.bin')):
    print(f"Checkpoint already exists at {ckpt1}, skipping Run 1.")
else:
    !python scripts/0b_train_predictor.py \
        --train_set ./data/train_predictor.jsonl \
        --output_dir ./checkpoints/opt125m-ltr-original \
        --loss_type listmle \
        --epochs 10

# Verify immediately -- don't wait until later to find out it failed
!ls {ckpt1}

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
PROJECT_DIR = '/content/drive/MyDrive/ltr_extension'
repo_path = os.path.join(PROJECT_DIR, 'prompt-aware-ltr')
%cd {repo_path}

import os

ckpt2 = os.path.join(repo_path, 'checkpoints', 'opt125m-ltr-marginloss')
if os.path.exists(os.path.join(ckpt2, 'pytorch_model.bin')):
    print(f"Checkpoint already exists at {ckpt2}, skipping Run 2.")
else:
    !python scripts/0b_train_predictor.py \
        --train_set ./data/train_predictor.jsonl \
        --output_dir ./checkpoints/opt125m-ltr-marginloss \
        --loss_type margin \
        --margin 1.0 \
        --pair_filter_delta 0.20 \
        --epochs 10

# Verify immediately
!ls {ckpt2}

## 10. FREE TIER ONLY — enable quantization

Skip / leave False if you're on Colab Pro with an A100. Only set True
on a free T4 (16GB) -- Llama-3.1-8B at FP16 alone needs ~16GB, leaving
no headroom for vLLM's KV cache.

In [ ]:
USE_QUANTIZATION = True   # set False on Colab Pro / A100

## 11. Run the benchmark sweep — Dataset 1 (in-distribution)

Compares all four schedulers: FCFS, Classification, Original LTR
(listmle checkpoint), and LTR + Prompt Length (margin-loss checkpoint
+ the prompt-length scoring patch). Expect 20-40 minutes depending on
GPU tier.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
PROJECT_DIR = '/content/drive/MyDrive/ltr_extension'
repo_path = os.path.join(PROJECT_DIR, 'prompt-aware-ltr')
%cd {repo_path}

# Pre-flight check -- fail fast with a clear message instead of burning
# GPU time on a run that's doomed to error out partway through.
ckpt1 = os.path.join(repo_path, 'checkpoints', 'opt125m-ltr-original', 'pytorch_model.bin')
ckpt2 = os.path.join(repo_path, 'checkpoints', 'opt125m-ltr-marginloss', 'pytorch_model.bin')
dataset1 = os.path.join(repo_path, 'data', 'dataset1_eval.jsonl')
assert os.path.exists(ckpt1), f"Missing checkpoint: {ckpt1} -- re-run Cell 18"
assert os.path.exists(ckpt2), f"Missing checkpoint: {ckpt2} -- re-run Cell 19"
assert os.path.exists(dataset1), f"Missing dataset: {dataset1} -- re-run Cell 16"
print("Pre-flight check passed -- both checkpoints and Dataset 1 found.")

quant_flag = "--quantize" if USE_QUANTIZATION else ""

!python scripts/3_vllm_benchmark.py \
    --model meta-llama/Llama-3.1-8B-Instruct \
    --ltr_predictor_path ./checkpoints/opt125m-ltr-original \
    --ltr_promptlen_predictor_path ./checkpoints/opt125m-ltr-marginloss \
    --dataset ./data/dataset1_eval.jsonl \
    --request_rates 5 10 20 30 40 50 60 \
    --schedulers fcfs classification ltr ltr_promptlen \
    --output results_dataset1.json \
    {quant_flag}

## 12. Run the benchmark sweep — Dataset 2 (out-of-distribution)

Reproduces the original paper's Dataset 2 evaluation (their reported
weak spot), now with your extension included as a fourth scheduler.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
PROJECT_DIR = '/content/drive/MyDrive/ltr_extension'
repo_path = os.path.join(PROJECT_DIR, 'prompt-aware-ltr')
%cd {repo_path}

ckpt1 = os.path.join(repo_path, 'checkpoints', 'opt125m-ltr-original', 'pytorch_model.bin')
ckpt2 = os.path.join(repo_path, 'checkpoints', 'opt125m-ltr-marginloss', 'pytorch_model.bin')
dataset2 = os.path.join(repo_path, 'data', 'dataset2_eval.jsonl')
assert os.path.exists(ckpt1), f"Missing checkpoint: {ckpt1} -- re-run Cell 18"
assert os.path.exists(ckpt2), f"Missing checkpoint: {ckpt2} -- re-run Cell 19"
assert os.path.exists(dataset2), f"Missing dataset: {dataset2} -- re-run Cell 16"
print("Pre-flight check passed -- both checkpoints and Dataset 2 found.")

quant_flag = "--quantize" if USE_QUANTIZATION else ""

!python scripts/3_vllm_benchmark.py \
    --model meta-llama/Llama-3.1-8B-Instruct \
    --ltr_predictor_path ./checkpoints/opt125m-ltr-original \
    --ltr_promptlen_predictor_path ./checkpoints/opt125m-ltr-marginloss \
    --dataset ./data/dataset2_eval.jsonl \
    --request_rates 5 10 20 30 40 50 60 \
    --schedulers fcfs classification ltr ltr_promptlen \
    --output results_dataset2.json \
    {quant_flag}

## 13. Confirm results saved (already in Drive — no extra copy step needed)

Since this entire notebook has been running inside
`/content/drive/MyDrive/ltr_extension/vllm-ltr` from Cell 3 onward,
`results_dataset1.json` and `results_dataset2.json` are ALREADY
sitting on your Drive. This cell just confirms that and shows you
the Drive path for downloading them to your Mac.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
PROJECT_DIR = '/content/drive/MyDrive/ltr_extension'
repo_path = os.path.join(PROJECT_DIR, 'vllm-ltr')
%cd {repo_path}

import os
for fname in ['results_dataset1.json', 'results_dataset2.json']:
    full_path = os.path.join(repo_path, fname)
    exists = os.path.exists(full_path)
    print(f"{fname}: {'FOUND' if exists else 'MISSING'} at {full_path}")

## 14. (Optional) Download results directly to your Mac

Only needed if you don't want to use Google Drive's web/desktop sync
to grab the files -- they're already safely in Drive either way.

In [ ]:
from google.colab import files
files.download('results_dataset1.json')
files.download('results_dataset2.json')

### (Optional) Paper-only comparison chart

Same two result files, but reproducing the original paper's exact Figure 3 —
only FCFS / Classification / LTR (no `ltr_promptlen`). Useful as a clean
side-by-side against the published baseline, separate from the 4-scheduler
chart `scripts/4_plot_results.py` produces further down.

In [ ]:
import json
import matplotlib.pyplot as plt
from collections import defaultdict

# Only the three schedulers from the original paper's Figure 3 —
# FCFS, Classification, and the original LTR predictor.
# Excludes ltr_promptlen (the new extension) entirely.
SCHEDULER_LABELS = {
    "fcfs": "FCFS",
    "classification": "Classification",
    "ltr": "LTR Predictor (Ours)",
}
SCHEDULER_COLORS = {
    "fcfs": "#1E5AA8",
    "classification": "#D97706",
    "ltr": "#0D9488",
}
SCHEDULER_STYLES = {
    "fcfs": "-o",
    "classification": "-s",
    "ltr": "-^",
}


def load_results(path):
    with open(path, "r") as f:
        return json.load(f)


def organize_by_scheduler(results):
    by_scheduler = defaultdict(list)
    for r in results:
        if r["scheduler"] not in SCHEDULER_LABELS:
            continue  # skip ltr_promptlen and anything else not in the original paper
        by_scheduler[r["scheduler"]].append((r["request_rate"], r["mean_latency_s_per_token"]))
    for k in by_scheduler:
        by_scheduler[k].sort(key=lambda x: x[0])
    return by_scheduler


def plot_on_axis(ax, by_scheduler, title):
    # Plot in a fixed order so the legend always lists FCFS, then
    # Classification, then LTR -- matching the original Figure 3 legend order.
    for scheduler_name in ["fcfs", "classification", "ltr"]:
        if scheduler_name not in by_scheduler:
            continue
        points = by_scheduler[scheduler_name]
        rates = [p[0] for p in points]
        latencies = [p[1] for p in points]
        label = SCHEDULER_LABELS[scheduler_name]
        color = SCHEDULER_COLORS[scheduler_name]
        style = SCHEDULER_STYLES[scheduler_name]
        ax.plot(rates, latencies, style, label=label, color=color, linewidth=2, markersize=6)
    ax.set_xlabel("Request rate (req/s)", fontsize=11)
    ax.set_ylabel("Latency (s/token)", fontsize=11)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.legend(loc="upper left", fontsize=9)
    ax.grid(True, alpha=0.3)


def print_summary_table(by_scheduler, dataset_label):
    print(f"\n=== {dataset_label} — mean latency (s/token) ===\n")
    all_rates = sorted({p[0] for points in by_scheduler.values() for p in points})
    header = "Rate".ljust(8) + "".join(SCHEDULER_LABELS[s].ljust(24) for s in ["fcfs", "classification", "ltr"] if s in by_scheduler)
    print(header)
    print("-" * len(header))
    for rate in all_rates:
        row = f"{rate:<8}"
        for s in ["fcfs", "classification", "ltr"]:
            if s not in by_scheduler:
                continue
            match = [v for r, v in by_scheduler[s] if r == rate]
            row += f"{match[0]:.4f}".ljust(24) if match else "—".ljust(24)
        print(row)


# --- Load both result files ---
results1 = load_results("results_dataset1.json")
results2 = load_results("results_dataset2.json")
by_sched1 = organize_by_scheduler(results1)
by_sched2 = organize_by_scheduler(results2)

# --- Print summary tables first ---
print_summary_table(by_sched1, "Dataset 1 (in-distribution) — original paper schedulers only")
print_summary_table(by_sched2, "Dataset 2 (out-of-distribution) — original paper schedulers only")

# --- Side-by-side chart, matching the original paper's Figure 3a/3b ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))
plot_on_axis(ax1, by_sched1, "(a) Dataset 1 (in-distribution)")
plot_on_axis(ax2, by_sched2, "(b) Dataset 2 (out-of-distribution)")
fig.suptitle("Llama-3.1-8B Instruct: Latency vs Request Rate\n(Original Paper Reproduction — FCFS / Classification / LTR)", fontsize=13, fontweight="bold")
fig.tight_layout()

fig.savefig("comparison_original_paper_only.png", dpi=200, bbox_inches="tight")
plt.show()

print("\nSaved comparison_original_paper_only.png — three-line reproduction of the original paper's Figure 3.")


---
## Optional — Kendall's Tau evaluation (can run here too, or later on Mac)

`2_kendalls_tau_eval.py` was originally designed to run on Mac M4
without a GPU, but since you're already set up here with both
checkpoints and the dataset, you can also run it directly in this
Colab session for convenience.

In [ ]:
!python scripts/2_kendalls_tau_eval.py \
    --predictor_path ./checkpoints/opt125m-ltr-original \
    --test_set ./data/dataset2_eval.jsonl \
    --alpha 0.7 --beta 0.3

---
## Known Issues (for reference — should not recur with this version)

| Problem | Root cause (from earlier sessions) | Fixed by |
|---|---|---|
| Lost checkpoint after restart | Working in `/content`, not Drive | Cell 1 mounts Drive first; everything runs inside `PROJECT_DIR` |
| `vllm==0.4.1` build fails | No Python 3.12 wheel for that version | Cell 4 pins `vllm==0.5.4` instead |
| `torch has no attribute float8_e8m0fnu` | `transformers` upgraded past what `torch` supports | Cell 4 pins `transformers==4.44.2` together with vllm in one install |
| `tokenizers>=0.19,<0.20 required, found 0.22.2` | Earlier `--no-deps` fix left tokenizers unpinned | Cell 4 pins `tokenizers==0.19.1` in the SAME command as transformers |
| `does not appear to have a file named config.json` | Wrong loader (`AutoModelForSequenceClassification`) for a custom architecture | `3_vllm_benchmark.py` / `2_kendalls_tau_eval.py` now reconstruct `LTRPredictor` directly and load `state_dict()` |
| `gradio requires starlette<1.0` warning | Harmless side-effect of installing transformers/tokenizers | Safe to ignore -- gradio is never used in this pipeline |
| `python3: can't open file '0b_train_predictor.py'` | Wrong working directory after restart | Cell 5 explicitly re-establishes `repo_path` and `cd`s into it |

## What success looks like

Two files at `vllm-ltr/results_dataset1.json` and
`results_dataset2.json`, each a list of records like:

```json
{
  "scheduler": "ltr_promptlen",
  "request_rate": 30.0,
  "mean_latency_s_per_token": 1.1,
  "p90_latency_s_per_token": 1.6,
  "n_requests": 850
}
```

Run `4_plot_results.py` on your Mac (or right here, see below) to turn
these into the comparison chart.

## (Optional) Generate the comparison charts right here in Colab

In [ ]:
!python scripts/4_plot_results.py \
    --input results_dataset1.json \
    --output comparison_dataset1.png \
    --dataset_label "Dataset 1 (in-distribution)"

!python scripts/4_plot_results.py \
    --input results_dataset2.json \
    --output comparison_dataset2.png \
    --dataset_label "Dataset 2 (out-of-distribution)"

In [ ]:
from IPython.display import Image, display
display(Image('comparison_dataset1.png'))
display(Image('comparison_dataset2.png'))